# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BarnaliBhowmik/FlyRank_ML/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### My response:
### Finding 1: Random Forest performed better than the baseline

The research paper reports that the Random Forest model achieved higher
Precision@50 than the baseline rules on the evaluated starter dataset.

**Methodology question:**  
I would ask how the positive label used for Precision@50 was created and
whether it represents a future outcome or a proxy derived from the same
information used to build the features.

This matters because a label derived from overlapping information can make the
model appear more effective than it would be when used at a real decision
moment.

This is a clarification I would want as a reader, rather than a conclusion
that the reported result is incorrect.

### Finding 2: Client-holdout validation was used

The research paper reports that validation kept whole clients out of training,
so the model was evaluated on clients that were not used for training.

**Methodology question:**  
I would ask whether the validation design matches the exact scope of the
research claim, including whether the same time period or other information
from the held-out clients could still overlap with the features or label.

A client-grouped split is a strong protection against client-level leakage,
but the time relationship between features and the target is also important
when the intended use is prediction or prioritization at a future decision
moment.

Overall, I would treat the reported findings as measured results for the
described dataset and validation design, while checking whether the same
evidence supports a broader claim.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [3]:
import pandas as pd
import numpy as np

print("Pandas and NumPy loaded successfully.")

Pandas and NumPy loaded successfully.


In [4]:
# Section 1 check: record the two methodology questions as simple audit items.

paper_audit = pd.DataFrame({
    "finding": [
        "Random Forest performed better than the baseline",
        "Client-holdout validation was used"
    ],
    "methodology_question": [
        "Where does the positive label come from, and does it overlap with feature information?",
        "Does the client-grouped validation design fully support the scope of the claim?"
    ]
})

paper_audit

,finding,methodology_question
0,Random Forest performed better than the baseline,"Where does the positive label come from, and d..."
1,Client-holdout validation was used,Does the client-grouped validation design full...


In [5]:
!pip -q install duckdb huggingface_hub pyarrow

import duckdb
import pandas as pd
import numpy as np
import pyarrow.dataset as ds

from huggingface_hub import HfFileSystem
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError("HF_TOKEN was not found in Colab Secrets.")

fs = HfFileSystem(token=HF_TOKEN)

print("Libraries loaded and Hugging Face access ready.")

Libraries loaded and Hugging Face access ready.


In [6]:
march_path = (
    "datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-03/"
)

march_files = fs.glob(march_path + "*.parquet")

print("March files found:", len(march_files))

March files found: 1


In [7]:
march_file = march_files[0]
march_uri = "hf://" + march_file

march_dataset = ds.dataset(
    march_uri,
    format="parquet",
    filesystem=fs
)

con = duckdb.connect()
con.register("march_data", march_dataset)

print("March 2026 data connected successfully.")

March 2026 data connected successfully.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [9]:
# Rebuild the page-level dataset used for the Week-5 model.

page_data_audit = con.sql("""
SELECT
    client_hash_id,
    content_hash_id,
    AVG(gsc_avg_position) AS gsc_avg_position,
    SUM(ga4_pageviews) AS ga4_pageviews,
    SUM(ga4_sessions) AS ga4_sessions,
    SUM(ga4_engaged_sessions) AS ga4_engaged_sessions,
    SUM(scroll_events) AS scroll_events,
    SUM(gsc_clicks) AS gsc_clicks,
    SUM(gsc_impressions) AS gsc_impressions
FROM march_data
WHERE gsc_data_available IS TRUE
  AND gsc_avg_position > 0
  AND gsc_avg_position <= 20
  AND gsc_impressions > 0
GROUP BY client_hash_id, content_hash_id
""").df()

page_data_audit["observed_ctr"] = (
    page_data_audit["gsc_clicks"] /
    page_data_audit["gsc_impressions"]
)

def position_bucket(position):
    if position <= 3:
        return "01-03"
    elif position <= 5:
        return "04-05"
    elif position <= 10:
        return "06-10"
    else:
        return "11-20"

page_data_audit["position_bucket"] = (
    page_data_audit["gsc_avg_position"].apply(position_bucket)
)

bucket_benchmarks = (
    page_data_audit
    .groupby("position_bucket")
    .apply(
        lambda x: x["gsc_clicks"].sum() / x["gsc_impressions"].sum()
    )
    .to_dict()
)

page_data_audit["benchmark_ctr"] = (
    page_data_audit["position_bucket"].map(bucket_benchmarks)
)

page_data_audit["ctr_gap"] = (
    page_data_audit["benchmark_ctr"] -
    page_data_audit["observed_ctr"]
).clip(lower=0)

print("Page-level rows:", len(page_data_audit))
print("Clients:", page_data_audit["client_hash_id"].nunique())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Page-level rows: 163911
Clients: 47


/tmp/ipykernel_4510/2218830705.py:44: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


In [10]:
from sklearn.model_selection import GroupShuffleSplit

# Split the final page-level dataset by client.
# This prevents pages from the same client appearing in both train and test.

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        page_data_audit,
        groups=page_data_audit["client_hash_id"]
    )
)

train_pages_audit = page_data_audit.iloc[train_idx].copy()
test_pages_audit = page_data_audit.iloc[test_idx].copy()

train_clients_audit = set(train_pages_audit["client_hash_id"])
test_clients_audit = set(test_pages_audit["client_hash_id"])

overlap = train_clients_audit.intersection(test_clients_audit)

print("Training rows:", len(train_pages_audit))
print("Test rows:", len(test_pages_audit))
print("Training clients:", len(train_clients_audit))
print("Test clients:", len(test_clients_audit))
print("Client overlap:", len(overlap))

if overlap:
    raise ValueError("Client leakage detected.")

print("Grouped split verified: no client appears in both sets.")

Training rows: 127831
Test rows: 36080
Training clients: 37
Test clients: 10
Client overlap: 0
Grouped split verified: no client appears in both sets.


In [11]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

features = [
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_sessions",
    "ga4_engaged_sessions",
    "scroll_events"
]

# Create the opportunity label using ONLY the training data.
# This prevents the test set from influencing the threshold.

ctr_gap_threshold_audit = train_pages_audit["ctr_gap"].median()

train_pages_audit["opportunity"] = (
    train_pages_audit["ctr_gap"] > ctr_gap_threshold_audit
).astype(int)

test_pages_audit["opportunity"] = (
    test_pages_audit["ctr_gap"] > ctr_gap_threshold_audit
).astype(int)

print("CTR-gap threshold:", ctr_gap_threshold_audit)
print("\nTraining opportunity rate:",
      train_pages_audit["opportunity"].mean())

print("Test opportunity rate:",
      test_pages_audit["opportunity"].mean())

CTR-gap threshold: 0.002926888731928216

Training opportunity rate: 0.2846883776235811
Test opportunity rate: 0.2521341463414634


In [12]:
model_audit = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("logistic_regression",
     LogisticRegression(max_iter=1000, random_state=42))
])

model_audit.fit(
    train_pages_audit[features],
    train_pages_audit["opportunity"]
)

test_pages_audit["ml_score"] = model_audit.predict_proba(
    test_pages_audit[features]
)[:, 1]

print("Logistic Regression trained successfully.")

Logistic Regression trained successfully.


In [13]:
# Apply the Week-4 baseline to exactly the same held-out test pages.

test_pages_audit["baseline_score"] = (
    test_pages_audit["ctr_gap"] *
    np.log1p(test_pages_audit["gsc_impressions"])
)

def precision_at_20_audit(df, score_column):
    top20 = (
        df.sort_values(score_column, ascending=False)
        .head(20)
    )
    return top20["opportunity"].mean()

honest_ml_precision = precision_at_20_audit(
    test_pages_audit,
    "ml_score"
)

honest_baseline_precision = precision_at_20_audit(
    test_pages_audit,
    "baseline_score"
)

print("Week-6 Logistic Regression Precision@20:",
      honest_ml_precision)

print("Week-4 baseline Precision@20 on same test set:",
      honest_baseline_precision)

Week-6 Logistic Regression Precision@20: 0.85
Week-4 baseline Precision@20 on same test set: 1.0


In [15]:
# Week-6 validation comparison

before_after = pd.DataFrame({
    "method": [
        "Logistic Regression — grouped client split",
        "Week-4 baseline — same held-out test set"
    ],
    "Precision@20": [
        honest_ml_precision,
        honest_baseline_precision
    ]
})

before_after

,method,Precision@20
0,Logistic Regression — grouped client split,0.85
1,Week-4 baseline — same held-out test set,1.00


### Validation result

Under the explicit client-grouped split, Logistic Regression achieved a
measured Precision@20 of 0.85, while the Week-4 baseline achieved 1.00 on the
same held-out test pages.

The baseline therefore ranked more pages that matched the opportunity proxy in
this evaluation.

However, this comparison has an important limitation: the opportunity label is
derived from `ctr_gap`, and the Week-4 baseline score also directly uses
`ctr_gap`. Therefore, the baseline has direct access to the same proxy used to
define the evaluation target.

I therefore do not interpret the 1.00 baseline score as evidence that the
baseline is generally superior to machine learning. Instead, this result
reveals that the current target definition makes the baseline comparison
partly circular.

The Logistic Regression result of 0.85 is best described as measured
performance under the grouped client split on the March 2026 development
data. A stronger future validation design and a target defined independently
of the baseline would be needed for a stronger model-vs-baseline claim.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [17]:
# Final feature-set leakage audit.

label_or_proxy_fields = [
    "ctr_gap",
    "observed_ctr",
    "gsc_clicks",
    "gsc_impressions",
    "opportunity"
]

context_fields = [
    "client_hash_id",
    "content_hash_id"
]

print("Final model features:")
for feature in features:
    print(" -", feature)

print("\nLabel/proxy fields excluded:")
for field in label_or_proxy_fields:
    print(" -", field)

print("\nContext/identifier fields excluded:")
for field in context_fields:
    print(" -", field)

feature_leaks = set(features).intersection(
    set(label_or_proxy_fields + context_fields)
)

print("\nFeature-set leakage check:", feature_leaks)

if feature_leaks:
    raise ValueError("Potential leakage found in final feature set.")

print("No listed label/proxy or identifier fields are model features.")

Final model features:
 - gsc_avg_position
 - ga4_pageviews
 - ga4_sessions
 - ga4_engaged_sessions
 - scroll_events

Label/proxy fields excluded:
 - ctr_gap
 - observed_ctr
 - gsc_clicks
 - gsc_impressions
 - opportunity

Context/identifier fields excluded:
 - client_hash_id
 - content_hash_id

Feature-set leakage check: set()
No listed label/proxy or identifier fields are model features.


In [18]:
# Deliberate leakage test:
# Create a fake feature that directly contains the opportunity label.

leak_test = test_pages_audit[
    ["opportunity"]
].copy()

leak_test["fake_leaked_feature"] = leak_test["opportunity"]

leaked_top20 = (
    leak_test
    .sort_values("fake_leaked_feature", ascending=False)
    .head(20)
)

leaked_precision = leaked_top20["opportunity"].mean()

print("Deliberate leaked-feature Precision@20:", leaked_precision)

# Remove the deliberately leaked feature.
leak_test = leak_test.drop(columns=["fake_leaked_feature"])

print("Deliberately leaked feature removed.")

Deliberate leaked-feature Precision@20: 1.0
Deliberately leaked feature removed.


### Leakage demonstration

The deliberately leaked feature produced an artificially perfect Precision@20
because it directly contained the opportunity label.

This demonstrates why label-derived fields must not be used as model features.
The final model feature set passed the leakage audit above, and the deliberately
leaked feature was removed after the demonstration.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### My response: Original bold claim

> Logistic Regression can identify pages with CTR improvement opportunities better
> than the rule-based baseline.

### Safer claim

> On the March 2026 development data, Logistic Regression showed measured
> Precision@20 of 0.85 under a client-grouped validation split, while the
> Week-4 baseline showed 1.00 on the same held-out pages. These results are
> directional and indicate that the current Logistic Regression model did not
> outperform the baseline for this opportunity proxy. The scoring approach
> should therefore be treated as decision-support for human review rather than
> as evidence that the model will improve CTR or generalize to future data.

### Why this wording is safer

- **Observed:** the results were observed on March 2026 data.
- **Measured:** Precision@20 was explicitly measured.
- **Directional:** the result is not treated as proof of future performance.
- **Decision-support:** the score is positioned as a prioritization aid for
  human review.
- **No causal claim:** the analysis does not show that acting on the ranked
  pages will increase CTR.

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.